# M04 · Earthquake catalogs: read, clean, look

**Week 1 · about 7.8 h (+ 1.5 h optional)** · before this: M01, M02

> **한국어 요약** 지진 목록에 무엇이 들어 있는지, 규모의 종류(ML, mb, Mw)가 어떻게 다른지 배우고 기상청 목록의 정밀도와 관측망 시기를 확인합니다. 규모-빈도 분포, UTC 시간 창 선택, 두 목록의 일대일 대조 함수를 직접 만듭니다. 시기별 완전성 규모를 그림에서 읽고, 규모 2.0 아래위의 지진 수를 해마다 비교합니다.

**You will**

- Count events per magnitude bin
- Select events in a UTC time window
- Pair the events that two catalogs share
- Read completeness by epoch; compare counts around ML 2.0

**New words:** frequency-magnitude distribution (규모-빈도 분포), magnitude of completeness (Mc; 완전성 규모), Gutenberg-Richter law (구텐베르크-리히터 법칙), magnitude type (규모 종류), network epoch (관측망 시기), time zero (기준 시각), catalog matching (지진 목록 대조), floating-point number (부동소수점 수) · [all terms](../../../docs/glossary.md)

**Optional reading:** Mignan & Woessner (2012), maximum curvature; Gutenberg & Richter (1944), four pages

Log any AI help in `work/ai_log/M04.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M04.yaml](../../../.sgtl/criteria/M04.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DM = 0.1                                   # magnitude bin width: the KMA list gives ML in steps of 0.1
SEQUENCE = "gyeongju2016"                  # a named sequence from st.data.sequence (Part 4)
FORESHOCK_UTC = "2016-09-12T10:44:32Z"     # the ML 5.1 foreshock of the Gyeongju sequence (KMA, UTC)
MAINSHOCK_UTC = "2016-09-12T11:32:54Z"     # the ML 5.8 mainshock of the Gyeongju sequence (KMA, UTC)
DAYS_BEFORE = 1.0                          # time window before time zero, in days (Part 4)
DAYS_AFTER = 30.0                          # time window after time zero, in days (Part 4)
PILEUP_YEARS = range(2008, 2027)           # UTC years for task 5: 2008 up to and including 2026
USE_NETWORK = True                         # False: never contact a data center (cached copies are still used)
NETWORK_TIMEOUT_S = 60                     # give up on a data center after this many seconds
DOWNLOAD_DIR = "downloads"                 # downloaded catalogs are cached here, next to this notebook
FIGURE_DIR = "figures"                     # your figure files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M04")

import io
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Path(DOWNLOAD_DIR).mkdir(exist_ok=True)
Path(FIGURE_DIR).mkdir(exist_ok=True)

## Part 1 · What a catalog holds, and who makes it (about 50 min)

An earthquake catalog has one row per earthquake: its origin time, location, depth and magnitude. Every agency
builds its own catalog from its own stations. KMA publishes the catalog of this course (KOGL Type 1: reuse it and
name the source). The **ISC** (International Seismological Centre) collects reports from many agencies; **USGS
ComCat** is global but lists only the larger Korean events. So one earthquake gets slightly different numbers in
each catalog. The KMA list:

In [ ]:
cat = st.data.kma_catalog()
print(f"KMA catalog: {len(cat)} events, {cat['origin_utc'].min():%Y-%m-%d} to {cat['origin_utc'].max():%Y-%m-%d} (UTC)")
print("columns:", list(cat.columns))
print(cat.attrs["source"].splitlines()[0])
cat.loc[cat["ml"] >= 5.0, ["event_id", "origin_utc", "latitude", "longitude", "depth_km", "ml"]]

A **magnitude type** says how a magnitude was measured. ML, the local magnitude, uses the largest swing on nearby
seismometers; KMA lists ML. mb, the **body-wave magnitude**, uses P waves recorded far away and stops growing near
magnitude 6. Mw, the **moment magnitude**, comes from the size of the rupture. For one earthquake the types can
differ by several tenths.

The KMA list is a CSV file: one row has room for one location and one magnitude. **QuakeML**, the standard XML
format for catalogs, gives an event several origins and magnitudes, each with its uncertainty and agency. The cell
writes the Gyeongju mainshock as QuakeML with ObsPy and reads it back.

**Watch out:** QuakeML gives depths in meters; the KMA list gives kilometers.

In [ ]:
from obspy import UTCDateTime, read_events
from obspy.core.event import Catalog, Event, Magnitude, Origin

main_row = cat.loc[cat["event_id"] == "2016000291"].iloc[0]
demo_origin = Origin(time=UTCDateTime(main_row["origin_utc"].isoformat()), latitude=float(main_row["latitude"]),
                     longitude=float(main_row["longitude"]), depth=float(main_row["depth_km"]) * 1000.0)  # meters
demo_event = Event(origins=[demo_origin], magnitudes=[Magnitude(mag=float(main_row["ml"]), magnitude_type="ML")])
buffer = io.BytesIO()
Catalog(events=[demo_event]).write(buffer, format="QUAKEML")
quakeml_text = buffer.getvalue().decode("utf-8")
start = quakeml_text.find("<origin")
print(f"{len(quakeml_text)} characters of QuakeML for one event; the origin part:")
print(quakeml_text[start:quakeml_text.find("</origin>") + 9])

again = read_events(io.BytesIO(buffer.getvalue()))
o = again[0].origins[0]
print("read back:", o.time, o.latitude, o.longitude, o.depth, "m depth,", again[0].magnitudes[0].magnitude_type)

**Asking a data center.** Many data centers answer the same web requests, the **FDSN** web services.
`Client("USGS").get_events(...)` asks ComCat for a time span (UTC), a box (degrees; west and south are negative) and
a smallest magnitude, and returns an ObsPy `Catalog`. The tool `events_table` below turns a `Catalog` into a table
with one row per event and depths in km.

In [ ]:
def events_table(events):
    """One row per event of an ObsPy Catalog, with its preferred (or else first) origin and magnitude.

    events  : obspy Catalog, for example from Client.get_events or read_events
    returns : pandas DataFrame with event_id (text), origin_utc (aware, UTC), latitude, longitude (degrees),
              depth_km, depth_err_km (km, NaN when not given), mag, mag_err, mag_type; sorted by origin_utc
    """
    rows = []
    for ev in events:
        org = ev.preferred_origin() or (ev.origins[0] if ev.origins else None)
        mag = ev.preferred_magnitude() or (ev.magnitudes[0] if ev.magnitudes else None)
        if org is None:
            continue
        found = re.search(r"eventid=([^&]+)", str(ev.resource_id))
        depth_m = org.depth if org.depth is not None else np.nan
        depth_err_m = org.depth_errors.uncertainty if org.depth_errors.uncertainty is not None else np.nan
        rows.append({
            "event_id": found.group(1) if found else str(ev.resource_id).rstrip("/").split("/")[-1],
            "origin_utc": pd.Timestamp(org.time.isoformat()).tz_localize("UTC"),
            "latitude": org.latitude,
            "longitude": org.longitude,
            "depth_km": depth_m / 1000.0,                 # QuakeML depths are in meters
            "depth_err_km": depth_err_m / 1000.0,
            "mag": mag.mag if mag is not None else np.nan,
            "mag_err": mag.mag_errors.uncertainty if mag is not None and mag.mag_errors.uncertainty else np.nan,
            "mag_type": mag.magnitude_type if mag is not None else None,
        })
    columns = ["event_id", "origin_utc", "latitude", "longitude", "depth_km", "depth_err_km", "mag", "mag_err",
               "mag_type"]
    table = pd.DataFrame(rows, columns=columns)
    table["origin_utc"] = pd.to_datetime(table["origin_utc"], utc=True)
    return table.sort_values("origin_utc", kind="stable").reset_index(drop=True)

The next cell is *optional*. It asks ComCat for the events of magnitude 2.5 or more around Korea in 2016-2017. It
reads the copy in `DOWNLOAD_DIR` if there is one, and contacts USGS only when `USE_NETWORK` is True.

In [ ]:
comcat_cache = Path(DOWNLOAD_DIR) / "comcat_korea_2016_2017.xml"
comcat_korea = None
if comcat_cache.is_file():
    comcat_korea = events_table(read_events(str(comcat_cache)))
    print(f"Read the cached copy {comcat_cache}.")
elif USE_NETWORK:
    try:
        from obspy.clients.fdsn import Client

        korea_events = Client("USGS", timeout=NETWORK_TIMEOUT_S).get_events(
            starttime=UTCDateTime("2016-01-01"), endtime=UTCDateTime("2018-01-01"), minlatitude=33.0,
            maxlatitude=39.0, minlongitude=124.0, maxlongitude=132.0, minmagnitude=2.5)
        korea_events.write(str(comcat_cache), format="QUAKEML")
        comcat_korea = events_table(korea_events)
        print(f"Downloaded {len(korea_events)} events from ComCat; saved {comcat_cache} for the next run.")
    except Exception as exc:   # no network, a slow server, ...: this step is optional
        print(f"Could not download from ComCat ({type(exc).__name__}). This step is optional; the rest of the "
              "notebook works without it. Run this cell again later to try once more.")
else:
    print("USE_NETWORK is False and there is no cached copy: skipping the optional ComCat download.")
if comcat_korea is not None:
    print(f"{len(comcat_korea)} ComCat events; magnitude types:", comcat_korea["mag_type"].value_counts().to_dict())
    display(comcat_korea[["event_id", "origin_utc", "latitude", "longitude", "depth_km", "depth_err_km", "mag",
                          "mag_type"]])

If the download worked, many depths are exactly 10 km: a round depth that the agency could not resolve and fixed by
hand.

## Part 2 · Precision and quirks of the KMA list (about 50 min)

Before you compute anything, find out how precise the numbers are and how the contents changed over time.
`data/catalogs/README.md` describes the file; the next two cells check the main points.

In [ ]:
years = cat["origin_utc"].dt.year                       # UTC years, one per event
tenths = cat["ml"].to_numpy() * 10.0
print("every ML is a multiple of 0.1:", bool(np.allclose(tenths, np.round(tenths))))
print("ML from", cat["ml"].min(), "to", cat["ml"].max())
known_depth = cat["depth_km"].dropna()
print("every given depth is a whole number of km:", bool(np.allclose(known_depth, np.round(known_depth))))
print("share of blank depths, by year:")
print(cat["depth_km"].isna().groupby(years).mean().loc[2007:2016].round(2).to_string())

In [ ]:
lat10, lon10 = cat["latitude"] * 10.0, cat["longitude"] * 10.0
on_tenth = pd.Series(np.isclose(lat10, lat10.round()) & np.isclose(lon10, lon10.round()), index=cat.index)
print("share of epicenters on a 0.1-degree grid, by year:")
print(on_tenth.groupby(years).mean().loc[2001:2008].round(2).to_string())

**Watch out:** the KMA list rounds. Magnitudes come in steps of 0.1 and depths in whole kilometers; depths are blank
for nearly all events before 2010 and for most in 2010-2011. Epicenters come in steps of 0.1° (about 11 km) before
2005 and of 0.01° (about 1 km) since. These are rounding steps, not error bars.

**Network epochs.** A denser network records smaller earthquakes. So we split the list into four **network epochs**:
1978-2007 (few stations), 2008-2015 (small events listed from 2008), 2016-2017 (the Gyeongju and Pohang sequences,
new stations) and 2018-2026 (the dense network). KMA calls events below ML 2.0 **microearthquakes** (미소지진).

In [ ]:
EPOCHS = {"1978-2007": (1978, 2007), "2008-2015": (2008, 2015), "2016-2017": (2016, 2017), "2018-2026": (2018, 2026)}

per_year = cat.groupby(years).size()
smallest = cat.groupby(years)["ml"].min()
fig, (ax_n, ax_m) = plt.subplots(2, 1, figsize=(7.0, 4.8), sharex=True)
ax_n.bar(per_year.index, per_year.to_numpy(), width=0.8, color="0.55")
ax_n.set_yscale("log")
ax_n.set_ylim(1, 20000)
ax_n.set_ylabel("Events per year")
ax_m.plot(smallest.index, smallest.to_numpy(), "o-", ms=3, color="tab:blue")
ax_m.set_ylabel(f"Smallest {st.plot.ML_LABEL} of the year")
ax_m.set_xlabel("Year (UTC)")
for label, (first, last) in EPOCHS.items():
    for ax in (ax_n, ax_m):
        ax.axvline(first - 0.5, color="tab:red", lw=0.8, ls="--")
    ax_n.text((first + last) / 2.0, 6000, label, ha="center", fontsize=8)
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "kma_events_per_year.png", dpi=150)

for label, (first, last) in EPOCHS.items():
    part = cat[(years >= first) & (years <= last)]
    print(f"{label}: {len(part):5d} events, smallest ML {part['ml'].min():.1f}, "
          f"{100.0 * (part['ml'] < 2.0).mean():4.1f} % below ML 2.0, {100.0 * part['depth_km'].notna().mean():5.1f} % with a depth")

**Not only South Korea.** The list also holds events in North Korea and under the seas, where only larger events are
recorded, less precisely. Choose a region and a period before you count. The map colors events by epoch:

In [ ]:
import pygmt

where_region = [122.5, 132.0, 32.0, 42.0]
epoch_colors = {"1978-2007": "black", "2008-2015": "dodgerblue", "2016-2017": "orange", "2018-2026": "red3"}
with st.plot.gmt_style(FONT_TITLE="12p,Helvetica"):
    fig_map = pygmt.Figure()
    fig_map.basemap(region=where_region, projection="M11c", frame=["af", "+tKMA catalog, 1978-2026"])
    fig_map.coast(land="gray92", water="white", shorelines="0.25p,gray40", borders="1/0.4p,gray50")
    for label, (first, last) in EPOCHS.items():
        part = cat[(years >= first) & (years <= last)]
        fig_map.plot(x=part["longitude"], y=part["latitude"], style="c0.05c", fill=epoch_colors[label],
                     label=f"{label}+S0.15c")
    fig_map.legend(position="jBR+o0.2c", box="+gwhite+p0.5p")
    fig_map.basemap(map_scale=st.plot.scale_bar_spec(where_region, km=200))
fig_map.show()
north = cat["latitude"] > 38.6
print(f"{north.sum()} events north of 38.6 N (North Korea and its seas); "
      f"{(cat['longitude'] > 130.0).sum()} events east of 130 E (East Sea)")

## Part 3 · The frequency-magnitude distribution (about 70 min, then task 1)

Small earthquakes are far more common than large ones. The **frequency-magnitude distribution** (FMD) counts them.
The non-cumulative FMD counts the events in each magnitude bin; the cumulative FMD counts those in each bin or
above, N(≥M).

KMA lists ML in steps of 0.1, the **magnitude bin width** Δm (`DM`). A listed 1.6 stands for 1.55 to 1.65: bins are
centered on multiples of Δm. Computers store decimals as **floating-point numbers**, which are often a tiny bit off:

In [ ]:
print(0.1 * 3, "| equal to 0.3?", 0.1 * 3 == 0.3)
print(1.6 / 0.1, "| 1.6 / 0.1 equal to 16?", 1.6 / 0.1 == 16)
print(1.5999999 / 0.1, 1.6000001 / 0.1)
noisy = np.array([1.5999999, 1.6000001, 1.6])
print("all three stand for 1.6:", noisy, "| exactly equal to 1.6:", noisy == 1.6)

### M04-T1 · Counting events per magnitude bin · core · about 60 min

Write `fmd(mags, dm=0.1)`. It returns `(centers, n_noncum, n_cum)`: the bin centers, the count in each bin and the
count in each bin or above. The centers run from the bin of the smallest magnitude to that of the largest, empty
bins included.

Example: `fmd([1.0, 1.0, 1.1, 1.3])` gives the centers 1.0, 1.1, 1.2, 1.3, `n_noncum` 2, 1, 0, 1 and `n_cum` 4, 2,
1, 1.

Not allowed: SeismoStats or `sgtl_tutorial` inside your function.

In [ ]:
# Before you code: for the magnitudes 2.0, 2.0, 2.1, 2.4 (dm = 0.1), what do you expect for the three arrays?
st.predict("M04-T1", """

""")

In [ ]:
def fmd(mags, dm=0.1):
    """Non-cumulative and cumulative frequency-magnitude distribution.

    mags    : 1-D list, numpy array or pandas Series of magnitudes in steps of dm (at least one value, no NaN);
              may be negative and may carry floating-point noise smaller than dm/1000
    dm      : float > 0, bin width (for example 0.05, 0.1 or 0.2); bins are centered on multiples of dm, with
              edges at center +- dm/2
    returns : (centers, n_noncum, n_cum), three 1-D numpy arrays of the same length: the bin centers in steps
              of dm from the bin of the smallest magnitude to that of the largest (empty bins included), the
              count in each bin, and the count in each bin or above (n_cum[0] is the total)
    """
    # Step 1: give every magnitude its bin number: the whole number k whose bin (k*dm - dm/2 to k*dm + dm/2)
    #         holds it. Think about what floating-point noise does here.
    # Step 2: count the magnitudes in every bin from the smallest k to the largest, empty bins included.
    # Step 3: the cumulative count of a bin adds its own count to the counts of all higher bins.
    # YOUR CODE HERE
    raise NotImplementedError("M04-T1: write your code here")

In [ ]:
st.check("M04-T1", func=fmd)

**Explain:** Into which bin would `np.floor(mags / dm)` put a magnitude stored as 1.5999999, and into which does
your `fmd` put it?

On a log scale, the cumulative counts above some magnitude fall on a straight line, the **Gutenberg-Richter law**:

$$\log_{10} N(\geq M) = a - b\,M$$

Its slope, the **b-value**, is close to 1 in most regions: ten times fewer events per magnitude unit. Your `fmd` on
the whole KMA list:

In [ ]:
centers_all, n_all, cum_all = fmd(cat["ml"], DM)
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.semilogy(centers_all, cum_all, "s", ms=3, color="0.35", label="Cumulative, N(≥M)")
ax.semilogy(centers_all, np.where(n_all > 0, n_all, np.nan), "o", ms=3.5, color="tab:blue", label="Per bin")
ax.set_xlabel(f"Local magnitude {st.plot.ML_LABEL}")
ax.set_ylabel("Number of events")
ax.set_title("KMA catalog, 1978-2026")
fig.legend(loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "kma_fmd_all.png", dpi=150)
print(f"{len(centers_all)} bins from {centers_all[0]:.1f} to {centers_all[-1]:.1f}; {cum_all[0]} events in all; "
      f"largest bin count {n_all.max()} at ML {centers_all[np.argmax(n_all)]:.1f}")

## Part 4 · Sequences and time zero (about 60 min, then task 2)

The 2016 Gyeongju sequence began with an ML 5.1 **foreshock**, an earthquake before the largest one, at 10:44:32 UTC
on 12 September; the ML 5.8 mainshock followed 48 minutes later. Times are counted from a **time zero**, the moment
chosen as t = 0. The plots use the mainshock as time zero. After task 2, a table compares the two shocks as time
zero.

In [ ]:
seq = st.data.sequence(SEQUENCE)
print(seq.attrs["sequence"]["description"], "-", len(seq), "events within",
      seq.attrs["sequence"]["radius_km"], "km of the center")
print(seq.nlargest(3, "ml")[["event_id", "origin_utc", "origin_kst", "ml"]].to_string(index=False))

t_main = pd.Timestamp(MAINSHOCK_UTC)
t_fore = pd.Timestamp(FORESHOCK_UTC)
since_days = ((seq["origin_utc"] - t_main) / pd.Timedelta(days=1)).to_numpy()
since_hours = since_days * 24.0
running = np.arange(1, len(seq) + 1)
fig, axes = plt.subplots(2, 2, figsize=(8.5, 5.2), sharex="col")
axes[0, 0].plot(since_days, seq["ml"], "o", ms=2.5, mfc="none", mec="tab:blue", mew=0.5)
axes[1, 0].step(np.sort(since_days), running, where="post", color="tab:blue")
early = (since_hours >= -2.0) & (since_hours <= 6.0)
axes[0, 1].plot(since_hours[early], seq["ml"].to_numpy()[early], "o", ms=3, mfc="none", mec="tab:blue", mew=0.6)
axes[1, 1].step(np.sort(since_hours[early]), np.arange(1, early.sum() + 1), where="post", color="tab:blue")
for ax in axes[:, 1]:
    ax.axvline((t_fore - t_main) / pd.Timedelta(hours=1), color="tab:red", lw=0.8, ls="--")
axes[0, 0].set_ylabel(st.plot.ML_LABEL)
axes[1, 0].set_ylabel("Cumulative number")
axes[1, 0].set_xlabel("Days since the mainshock")
axes[1, 1].set_xlabel("Hours since the mainshock (dashed: foreshock)")
axes[0, 0].set_title("Whole sequence")
axes[0, 1].set_title("First hours")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "gyeongju_time_magnitude_cumulative.png", dpi=150)
print(f"{early.sum()} events from 2 h before to 6 h after the mainshock; "
      f"foreshock at {(t_fore - t_main) / pd.Timedelta(minutes=1):.1f} min")

**Watch out:** times come in many spellings: `"2016-09-12T11:32:54Z"` (UTC), `"2016-09-12T20:32:54+09:00"` (the same
moment in KST, UTC + 9 h), a pandas `Timestamp`, or a time without a zone, which this course reads as UTC. Turn each
into one UTC moment before you compare it with `origin_utc`.

### M04-T2 · Events around a chosen moment · core · about 60 min

Write `select_window(df, t0_utc, days_before, days_after)`. It returns the rows of `df` from `days_before` days
before the time zero `t0_utc` to `days_after` days after it, both ends included, sorted by `origin_utc`. The
docstring lists the spellings `t0_utc` may come in.

Example: `select_window(seq, "2016-09-12T20:32:54+09:00", 0, 0)` returns one row: the ML 5.8 mainshock.

In [ ]:
# Before you code: about how many events do you expect in the first day after the mainshock?
st.predict("M04-T2", """

""")

In [ ]:
def select_window(df, t0_utc, days_before, days_after):
    """Events from days_before days before a time zero to days_after days after it, earliest first.

    df          : pandas DataFrame with at least origin_utc (time-zone-aware, UTC); not changed
    t0_utc      : time zero as text, a pandas Timestamp or a datetime; with a zone or an offset ("...Z",
                  "...+09:00") it is that moment, without one it is UTC
    days_before : float >= 0, days before time zero (fractions allowed)
    days_after  : float >= 0, days after time zero (fractions allowed)
    returns     : pandas DataFrame with exactly the columns of df: the rows with
                  t0 - days_before <= origin_utc <= t0 + days_after, sorted by origin_utc (earliest first);
                  an empty DataFrame with the same columns when no event is inside
    """
    # Step 1: turn t0_utc into one time-zone-aware moment in UTC, whatever spelling it came in.
    # Step 2: the two ends of the window, then keep the rows between them (ends included) and sort them.
    # YOUR CODE HERE
    raise NotImplementedError("M04-T2: write your code here")

In [ ]:
st.check("M04-T2", func=select_window)

In [ ]:
counts = []
for name, zero_text in (("foreshock", FORESHOCK_UTC), ("mainshock", MAINSHOCK_UTC)):
    zero = pd.Timestamp(zero_text)
    window = select_window(seq, zero_text, DAYS_BEFORE, DAYS_AFTER)
    after = window["origin_utc"] > zero
    counts.append({"time zero": name, "events in window": len(window),
                   "before time zero": int((window["origin_utc"] < zero).sum()),
                   "after time zero": int(after.sum()),
                   "first hour after": int((after & (window["origin_utc"] <= zero + pd.Timedelta(hours=1))).sum()),
                   "first day after": int((after & (window["origin_utc"] <= zero + pd.Timedelta(days=1))).sum())})
pd.DataFrame(counts)

**Explain:** Which time zero would you use to count the aftershocks of the ML 5.8 mainshock? Use the table above.

## Part 5 · The same earthquake in two catalogs (about 30 min, then task 3)

**Catalog matching** pairs the rows of two catalogs that describe the same earthquake. Two events are candidates
when their origin times differ by at most `max_dt_s` seconds and their epicenters by at most `max_dist_km` km. Each
event pairs with at most one event of the other catalog. `epicentral_km` gives the **great-circle distance**, the
shortest way along the surface, in km:

In [ ]:
from obspy.geodetics import locations2degrees

KM_PER_DEGREE = 6371.0 * np.pi / 180.0


def epicentral_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km (sphere of radius 6371 km) between points given in degrees."""
    arc = locations2degrees(np.asarray(lat1, dtype=float), np.asarray(lon1, dtype=float),
                            np.asarray(lat2, dtype=float), np.asarray(lon2, dtype=float))
    return np.asarray(arc, dtype=float) * KM_PER_DEGREE


print(f"Gyeongju to Pohang mainshock: {float(epicentral_km(35.76, 129.19, 36.11, 129.37)):.1f} km")

### M04-T3 · Pairing two catalogs · core · about 120 min

Write `match_events(cat_a, cat_b, max_dt_s, max_dist_km)`. It returns one row per kept pair: `id_a`, `id_b`, `dt_s`
and `dist_km`. Go through the candidates from the smallest |t_b − t_a| up; keep a pair only when neither event is in
a kept pair yet.

Example: event a at 36.25 N, 128.10 E; b1 3 s earlier at 36.26 N; b2 5 s later at the place of a. With 10 s and 20
km, a pairs with b1: `dt_s` = −3.0, `dist_km` ≈ 1.11.

In [ ]:
# Before you code: how far apart do you expect the KMA and USGS origin times and epicenters of the Gyeongju
# mainshock to be?
st.predict("M04-T3", """

""")

In [ ]:
def match_events(cat_a, cat_b, max_dt_s, max_dist_km):
    """One-to-one pairs of events shared by two catalogs, closest in time first.

    cat_a, cat_b : pandas DataFrames with event_id (text), origin_utc (aware, UTC), latitude, longitude
                   (degrees); rows in any order; not changed
    max_dt_s     : float > 0, largest allowed |t_b - t_a| in seconds (b may come before or after a)
    max_dist_km  : float > 0, largest allowed epicentral distance in km (both limits included)
    returns      : pandas DataFrame with the columns id_a, id_b, dt_s (t_b - t_a in seconds, negative when b
                   came first) and dist_km, one row per kept pair (rows in any order); with no pair, an empty
                   DataFrame with these four columns
    """
    # Step 1: every candidate pair: all combinations of a and b, with their time difference and distance,
    #         kept when both are within the limits.
    # Step 2: go through the candidates from the smallest |time difference| up, and keep a pair only when
    #         neither event has a partner yet.
    # YOUR CODE HERE
    raise NotImplementedError("M04-T3: write your code here")

In [ ]:
st.check("M04-T3", func=match_events)

In [ ]:
if comcat_korea is None:
    print("No ComCat table (no network and no cached copy), so there is nothing to compare. This step is optional.")
else:
    kma_1617 = cat[(cat["origin_utc"] >= pd.Timestamp("2016-01-01", tz="UTC"))
                   & (cat["origin_utc"] < pd.Timestamp("2018-01-01", tz="UTC"))]
    shared = match_events(kma_1617, comcat_korea, 10.0, 30.0)
    shared = shared.merge(kma_1617[["event_id", "ml"]], left_on="id_a", right_on="event_id").drop(columns="event_id")
    shared = shared.merge(comcat_korea[["event_id", "mag", "mag_type", "depth_km"]], left_on="id_b",
                          right_on="event_id").drop(columns="event_id")
    print(f"{len(shared)} of {len(comcat_korea)} ComCat events have a KMA partner "
          f"(KMA lists {len(kma_1617)} events in 2016-2017)")
    display(shared.round(2))

**Explain:** Why should the smaller time difference, not the smaller distance, decide between two candidates? Use
the table above if the download worked.

## Part 6 · Completeness changes with the network (about 40 min, then task 4)

A catalog misses small events whose waves reach too few stations. The **magnitude of completeness** Mc is the
smallest magnitude above which it lists almost every earthquake. Above Mc the per-bin counts fall steadily; below it
they drop. A quick estimate is **maximum curvature**, the bin with the most events, plus 0.2 because the peak alone
tends to be too low. Your `fmd` draws one panel per epoch:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8.5, 6.0), sharex=True)
for ax, (label, (first, last)) in zip(axes.ravel(), EPOCHS.items()):
    part = cat[(years >= first) & (years <= last)]
    centers_e, n_e, cum_e = fmd(part["ml"], DM)
    ax.semilogy(centers_e, cum_e, "s", ms=2.5, color="0.4", label="Cumulative, N(≥M)")
    ax.semilogy(centers_e, np.where(n_e > 0, n_e, np.nan), "o", ms=3.5, color="tab:blue", label="Per bin")
    ax.set_title(f"{label}: {len(part)} events")
    ax.grid(True, which="major", lw=0.3, color="0.85")
    ax.set_xticks(np.arange(0.0, 6.5, 0.5), minor=True)
for ax in axes[1]:
    ax.set_xlabel(f"Local magnitude {st.plot.ML_LABEL} (bins of {DM:g})")
for ax in axes[:, 0]:
    ax.set_ylabel("Number of events")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(Path(FIGURE_DIR) / "kma_fmd_by_epoch.png", dpi=150)

### M04-T4 · Completeness by epoch · core · about 40 min

Read Mc for each network epoch from the panels: the tallest blue circle (count per bin) plus 0.2. If unsure, print
your `fmd` counts for that epoch. Set `mc_by_eye`: a dict with the four keys of `EPOCHS`, one number each. The
check runs your `fmd` on cases of task 1 again, then accepts each value within 0.3 of this rule applied to your own
`fmd`.

Example: a tallest circle at ML 0.8 gives Mc 1.0.

In [ ]:
# Before you code: which epoch do you expect to have the highest Mc?
st.predict("M04-T4", """

""")

In [ ]:
# Shape of the answer: mc_by_eye = {"1978-2007": <Mc>, "2008-2015": <Mc>, "2016-2017": <Mc>, "2018-2026": <Mc>}
# Each value is a number (ML), not text. The check also takes your fmd of task 1, which drew the panels.
# YOUR CODE HERE
raise NotImplementedError("M04-T4: write your code here")

In [ ]:
st.check("M04-T4", answer={"mc_by_eye": mc_by_eye, "fmd": fmd})

**Explain:** Above Mc, the per-bin counts should fall steadily. Do they in the 2018-2026 panel near ML 2.0?

## Part 7 · Something odd near one magnitude (about 45 min, then task 5)

To measure what you saw, year by year, compare the bins just below ML 2.0 (1.8 and 1.9) with those just above it
(2.0 and 2.1). Under a Gutenberg-Richter law, the low bins hold $10^{0.2b}$ times the events of the high bins: about
1.6 for b = 1.

### M04-T5 · Something odd near one magnitude · core · about 60 min

Write `pileup_ratio(df, years, low=(1.8, 1.9), high=(2.0, 2.1))`. For each UTC year in `years`, it returns N(low) /
N(high): the events in the `low` bins divided by those in the `high` bins. An event counts for a bin when its ML
lies within ±0.05 of it (1.8999999 counts for 1.9).

Example: events in 2020 of ML 1.8, 1.9, 1.9, 2.0 and 2.1, and none in 2021, give 1.5 for 2020 and NaN for 2021.

In [ ]:
# Before you code: what value do you expect for the ratio in a typical recent year of the KMA list?
st.predict("M04-T5", """

""")

In [ ]:
def pileup_ratio(df, years, low=(1.8, 1.9), high=(2.0, 2.1)):
    """Per UTC year: the number of events in the low bins divided by the number in the high bins.

    df      : pandas DataFrame with origin_utc (aware, UTC) and ml (steps of 0.1, may carry tiny noise); rows in
              any order; not changed
    years   : whole-number years to report (list, range or array), counted in UTC
    low     : tuple of one or more magnitude bins (multiples of 0.1) for the numerator
    high    : tuple of one or more magnitude bins (multiples of 0.1) for the denominator
    returns : pandas Series of floats indexed by years (in the order given): N(low) / N(high); NaN when
              N(high) is 0 (also in a year without events), 0.0 when only N(low) is 0
    """
    # Step 1: put every magnitude into its 0.1 bin, robust to floating-point noise.
    # Step 2: count, for each year, the events in the low bins and in the high bins (years in UTC).
    # Step 3: divide, with NaN where the high count is zero, and keep the years in the order given.
    # YOUR CODE HERE
    raise NotImplementedError("M04-T5: write your code here")

In [ ]:
st.check("M04-T5", func=pileup_ratio)

In [ ]:
kma_ratio = pileup_ratio(cat, PILEUP_YEARS)
practice = st.synth.synthetic_catalog(np.random.default_rng(2026), 300_000, start="2008-01-01", end="2027-01-01",
                                      b=1.0, mc=0.5)   # made-up Gutenberg-Richter catalog, b = 1, no oddity
practice_ratio = pileup_ratio(practice, PILEUP_YEARS)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(kma_ratio.index, kma_ratio.to_numpy(), "o-", ms=4, color="tab:red", label="KMA catalog")
ax.plot(practice_ratio.index, practice_ratio.to_numpy(), "s--", ms=3, color="0.5",
        label="Made-up catalog, b = 1")
ax.set_xlabel("Year (UTC)")
ax.set_ylabel("N(ML 1.8-1.9) / N(ML 2.0-2.1)")
fig.legend(loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(Path(FIGURE_DIR) / "kma_ratio_by_year.png", dpi=150)
pd.DataFrame({"KMA": kma_ratio.round(2), "made-up, b = 1": practice_ratio.round(2)})

**Explain:** The made-up catalog follows a Gutenberg-Richter law with b = 1. In which years does the KMA ratio
differ most from it, and by how much?

**Watch out:** in most years since 2008 the KMA counts step down at ML 2.0. Never fit one b-value to magnitudes on
both sides of ML 2.0: start above it (M05 uses ML 2.1).

## Part 8 · Stretch: Ridgecrest 2019 from ComCat (about 90 min, optional)

The 2019 Ridgecrest sequence in California had an M 7.1 mainshock on 6 July (UTC). ComCat lists its magnitudes in
steps of 0.01, of several types. Its FMD beside that of Gyeongju shows what changes between catalogs.

### M04-T6 · Ridgecrest and Gyeongju side by side · stretch · about 90 min

Write `ridgecrest_events(cache_csv)`. It reads `cache_csv` when it exists; otherwise it asks ComCat for 35.4 to 36.1
N, 117.9 to 117.2 W, 2019-07-04 to 2019-08-04 UTC, magnitude 2.5 or more, and saves the `events_table` of the answer
there. Then draw both FMDs in one figure, save it as a PNG and set `ridgecrest_files`. The check reads only these
files: at least 1000 events inside the request, depths in km, a figure newer than the table.

In [ ]:
# Before you code: about how many Ridgecrest events of magnitude 2.5 or more do you expect in the first month?
st.predict("M04-T6", """

""")

In [ ]:
def ridgecrest_events(cache_csv):
    """Ridgecrest 2019 events (35.4-36.1 N, 117.9-117.2 W, 2019-07-04 to 2019-08-04 UTC, magnitude >= 2.5).

    cache_csv : str or Path of the CSV cache: read when it exists (origin_utc back to aware UTC times);
                otherwise written with to_csv(index=False) after a successful download
    returns   : pandas DataFrame like events_table() gives (origin_utc aware, UTC; depths in km), or None
                when there is no cache and no download (USE_NETWORK is False, or the request failed: then
                print one short sentence); ask ComCat with the timeout NETWORK_TIMEOUT_S
    """
    # Step 1: the cached copy, if there is one.
    # Step 2: otherwise the request to ComCat (guarded: a failure prints a sentence and returns None).
    # Step 3: save the new table, so the next run does not need the network.
    # YOUR CODE HERE
    raise NotImplementedError("M04-T6: write your code here")

In [ ]:
# Call ridgecrest_events with a cache path in DOWNLOAD_DIR; draw both FMDs in one figure (Gyeongju column "ml",
# Ridgecrest column "mag"); save it as a PNG in FIGURE_DIR after the table; print the number of events of each
# catalog and the Ridgecrest magnitude types. Then set ridgecrest_files = {"csv": <table path>, "figure": <PNG path>}.
# Keep the figure working when the Ridgecrest table is None (no network).
# YOUR CODE HERE
raise NotImplementedError("M04-T6: write your code here")

In [ ]:
st.check("M04-T6", answer=ridgecrest_files)

**Explain:** Could you compare a b-value of the Ridgecrest table directly with one of Gyeongju? Use the printed
magnitude types.

## Wrap-up

Your progress in this module:

In [ ]:
st.progress("M04")

**Read more:** Mignan and Woessner (2012), the magnitude of completeness; Bormann and Dewey (2014), magnitude
standards; Woo et al. (2019), the Gyeongju sequence. The [reading list](../../../docs/reading_list.md) has the rest.

Next: M05 (statistical seismology). Commit your notebook as in M00.